# Task 16 — Hyperparameter Tuning using GridSearchCV

Dataset: Breast Cancer (sklearn)

Objective:
- Train baseline model
- Tune hyperparameters using GridSearchCV
- Use cross-validation
- Compare tuned vs default performance


In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [13]:
# Core libraries
import pandas as pd
import numpy as np

# Sklearn datasets and model tools
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Model
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [14]:
# Load dataset from sklearn
data = load_breast_cancer()

# Convert to DataFrame for inspection
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Dataset loaded successfully")
print("Feature shape:", X.shape)
print("Target shape :", y.shape)

# Show first rows
X.head()


Dataset loaded successfully
Feature shape: (569, 30)
Target shape : (569,)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [15]:
# Check class balance
print("Class distribution:")
print(pd.Series(y).value_counts())


Class distribution:
1    357
0    212
Name: count, dtype: int64


In [16]:
# Split data — stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:", X_train.shape)
print("Testing size :", X_test.shape)


Training size: (455, 30)
Testing size : (114, 30)


In [17]:
# Build pipeline so scaling happens inside CV (prevents leakage)
baseline_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC())
])

print("Baseline pipeline created")


Baseline pipeline created


In [18]:
# Train baseline model
baseline_pipeline.fit(X_train, y_train)

print("Baseline model trained")


Baseline model trained


In [19]:
# Predict baseline
y_pred_base = baseline_pipeline.predict(X_test)

# Compute metrics
base_acc = accuracy_score(y_test, y_pred_base)
base_prec = precision_score(y_test, y_pred_base)
base_rec = recall_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base)

print("Baseline Model Metrics")
print("Accuracy :", base_acc)
print("Precision:", base_prec)
print("Recall   :", base_rec)
print("F1 Score :", base_f1)


Baseline Model Metrics
Accuracy : 0.9824561403508771
Precision: 0.9861111111111112
Recall   : 0.9861111111111112
F1 Score : 0.9861111111111112


In [20]:
# Define parameter grid for tuning
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": [0.001, 0.01, 0.1, 1],
    "svm__kernel": ["rbf"]
}

print("Parameter grid defined")


Parameter grid defined


In [21]:
# Setup GridSearch with cross-validation
grid_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    cv=5,                 # 5-fold cross validation
    scoring="f1",         # good metric for classification
    n_jobs=-1             # use all cores
)

print("GridSearchCV configured")


GridSearchCV configured


In [22]:
# Fit grid search
grid_search.fit(X_train, y_train)

print("Grid search completed")


Grid search completed


In [23]:
print("Best Parameters Found:")
print(grid_search.best_params_)

print("Best Cross-Validation Score:")
print(grid_search.best_score_)


Best Parameters Found:
{'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Best Cross-Validation Score:
0.9844323436002114


In [24]:
# Get best model
best_model = grid_search.best_estimator_

# Predict tuned model
y_pred_tuned = best_model.predict(X_test)

# Metrics
tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_prec = precision_score(y_test, y_pred_tuned)
tuned_rec = recall_score(y_test, y_pred_tuned)
tuned_f1 = f1_score(y_test, y_pred_tuned)

print("Tuned Model Metrics")
print("Accuracy :", tuned_acc)
print("Precision:", tuned_prec)
print("Recall   :", tuned_rec)
print("F1 Score :", tuned_f1)


Tuned Model Metrics
Accuracy : 0.9824561403508771
Precision: 0.9861111111111112
Recall   : 0.9861111111111112
F1 Score : 0.9861111111111112


In [25]:
comparison = pd.DataFrame([
    ["Default SVM", base_acc, base_prec, base_rec, base_f1],
    ["Tuned SVM", tuned_acc, tuned_prec, tuned_rec, tuned_f1]
], columns=["Model", "Accuracy", "Precision", "Recall", "F1"])

comparison


,Model,Accuracy,Precision,Recall,F1
0,Default SVM,0.982456,0.986111,0.986111,0.986111
1,Tuned SVM,0.982456,0.986111,0.986111,0.986111


In [26]:
import joblib

joblib.dump(best_model, "tuned_svm_pipeline.pkl")

print("Tuned pipeline saved as tuned_svm_pipeline.pkl")


Tuned pipeline saved as tuned_svm_pipeline.pkl
